# 🛡️ Sentinel: Fine-tuning Llama-3-8B for Vulnerability Detection

This notebook fine-tunes Llama-3-8B using **Unsloth** on a **T4 GPU** (Google Colab Free Tier).

**Memory-optimized for 16GB VRAM:**
- 4-bit quantization (~5GB model)
- LoRA adapters (99.7% fewer trainable params)
- Gradient checkpointing (60% less activation memory)

**What we'll do:**
1. Install Unsloth + xformers
2. Load Llama-3-8B in 4-bit
3. Format vulnerability data (Instruction → Buggy Code → Fixed Code)
4. Train with LoRA
5. Save adapters to Google Drive

## 📦 Step 1: Installation

Install Unsloth optimized for Colab's T4 GPU.

In [ ]:
%%capture
# Install Unsloth - optimized for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Verify GPU and memory
import torch
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 🦙 Step 2: Load Llama-3-8B (4-bit Quantized)

Using Unsloth's pre-quantized model saves ~70% memory:
- Full model: ~16GB → **Won't fit on T4**
- 4-bit quantized: ~5GB → **Fits perfectly!**

In [ ]:
from unsloth import FastLanguageModel

# Configuration
MAX_SEQ_LENGTH = 2048  # Supports up to 8192, but 2048 saves memory
DTYPE = None  # Auto-detect (float16 for T4)
LOAD_IN_4BIT = True  # Critical for T4 (16GB VRAM)

# Load the 4-bit quantized model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",  # Pre-quantized for speed
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"✅ Model loaded! Vocab size: {len(tokenizer)}")

## 🔧 Step 3: Configure LoRA Adapters

**LoRA (Low-Rank Adaptation)** freezes the base model and trains small adapters.

This reduces trainable parameters from **8B → ~20M** (99.7% reduction!)

**Key parameters:**
- `r=16`: Rank of the low-rank matrices (higher = more capacity, more memory)
- `lora_alpha=16`: Scaling factor (usually = r)
- `target_modules`: Which layers to adapt (attention projections are most effective)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank: 8-64 typical, 16 is good balance
    lora_alpha=16,  # Scaling factor
    lora_dropout=0,  # 0 is optimized by Unsloth
    target_modules=[
        "q_proj",   # Query projection
        "k_proj",   # Key projection
        "v_proj",   # Value projection
        "o_proj",   # Output projection
        "gate_proj",  # MLP gate
        "up_proj",    # MLP up
        "down_proj",  # MLP down
    ],
    bias="none",  # Don't train biases (saves memory)
    use_gradient_checkpointing="unsloth",  # 60% less VRAM!
    random_state=42,
    use_rslora=False,  # Rank-stabilized LoRA (optional)
    loftq_config=None,
)

# Check trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"🎯 Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

## 📊 Step 4: Prepare Vulnerability Dataset

We format data as:
```
### Instruction:
Analyze the following code for security vulnerabilities and provide a fixed version.

### Input (Buggy Code):
<vulnerable code here>

### Output (Fixed Code):
<patched code here>
```

In [ ]:
# Define the prompt template
VULNERABILITY_PROMPT = """### Instruction:
{instruction}

### Input (Buggy Code):
{buggy_code}

### Output (Fixed Code):
{fixed_code}"""

# EOS token to mark end of generation
EOS_TOKEN = tokenizer.eos_token

def format_vulnerability_sample(sample: dict) -> dict:
    """
    Format a sample into the vulnerability detection prompt format.
    
    Expected input format (adapt based on your dataset):
    - 'vulnerable_code' or 'buggy_code': The code with vulnerability
    - 'fixed_code' or 'patched_code': The corrected code
    - 'cwe_id' or 'vulnerability_type': (Optional) Type of vulnerability
    """
    # Extract fields (adapt these keys to your dataset)
    buggy = sample.get('vulnerable_code') or sample.get('buggy_code') or sample.get('func_before', '')
    fixed = sample.get('fixed_code') or sample.get('patched_code') or sample.get('func_after', '')
    vuln_type = sample.get('cwe_id') or sample.get('vulnerability_type') or 'security vulnerability'
    
    instruction = f"Analyze the following code for {vuln_type} and provide a fixed version."
    
    formatted_text = VULNERABILITY_PROMPT.format(
        instruction=instruction,
        buggy_code=buggy.strip(),
        fixed_code=fixed.strip()
    ) + EOS_TOKEN
    
    return {"text": formatted_text}

### Load a Vulnerability Dataset

**Options:**
1. **CVEFixes** - Real CVE patches (recommended)
2. **BigVul** - Large vulnerability dataset
3. **Custom** - Your own dataset

Below we use example data for testing. **Replace with your dataset for production!**

In [ ]:
from datasets import load_dataset, Dataset

# ============================================
# OPTION 1: Use a real vulnerability dataset
# ============================================
# Uncomment ONE of these for production training:

# CVEFixes dataset (if available)
# dataset = load_dataset("ahmed-masry/CVEFixes", split="train")

# BigVul dataset
# dataset = load_dataset("benjis/bigvul_dataset", split="train")

# Code-fix dataset
# dataset = load_dataset("composite-hl/code-fix", split="train[:5000]")

# ============================================
# OPTION 2: Example data for testing
# ============================================
EXAMPLE_DATA = [
    {
        "vulnerable_code": '''def login(username, password):
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    cursor.execute(query)
    return cursor.fetchone()''',
        "fixed_code": '''def login(username, password):
    query = "SELECT * FROM users WHERE username=? AND password=?"
    cursor.execute(query, (username, password))
    return cursor.fetchone()''',
        "cwe_id": "CWE-89 (SQL Injection)"
    },
    {
        "vulnerable_code": '''import os
def run_command(user_input):
    os.system(f"echo {user_input}")''',
        "fixed_code": '''import subprocess
import shlex
def run_command(user_input):
    subprocess.run(["echo", shlex.quote(user_input)], check=True)''',
        "cwe_id": "CWE-78 (OS Command Injection)"
    },
    {
        "vulnerable_code": '''def read_file(filename):
    path = "/var/www/files/" + filename
    with open(path, 'r') as f:
        return f.read()''',
        "fixed_code": '''import os
def read_file(filename):
    base_path = "/var/www/files/"
    full_path = os.path.normpath(os.path.join(base_path, filename))
    if not full_path.startswith(base_path):
        raise ValueError("Invalid path")
    with open(full_path, 'r') as f:
        return f.read()''',
        "cwe_id": "CWE-22 (Path Traversal)"
    },
    {
        "vulnerable_code": '''from flask import request
@app.route('/search')
def search():
    query = request.args.get('q')
    return f"<h1>Results for: {query}</h1>"''',
        "fixed_code": '''from flask import request, escape
@app.route('/search')
def search():
    query = request.args.get('q')
    return f"<h1>Results for: {escape(query)}</h1>"''',
        "cwe_id": "CWE-79 (Cross-Site Scripting)"
    },
    {
        "vulnerable_code": '''import pickle
def load_data(data_bytes):
    return pickle.loads(data_bytes)''',
        "fixed_code": '''import json
def load_data(data_bytes):
    return json.loads(data_bytes.decode('utf-8'))''',
        "cwe_id": "CWE-502 (Deserialization of Untrusted Data)"
    },
]

# Create dataset
dataset = Dataset.from_list(EXAMPLE_DATA)
print(f"📚 Dataset size: {len(dataset)} samples")

# Format the dataset
dataset = dataset.map(format_vulnerability_sample)

# Preview a sample
print("\n📝 Sample formatted prompt:")
print("-" * 50)
print(dataset[0]["text"])

## 🏋️ Step 5: Training Configuration

**Memory-optimized settings for T4 GPU:**

| Setting | Value | Purpose |
|---------|-------|---------|
| `batch_size` | 2 | Fits in 16GB VRAM |
| `gradient_accumulation` | 4 | Effective batch = 8 |
| `optim` | adamw_8bit | 50% less memory |
| `fp16` | True | Native T4 support |

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for T4 (16GB)
training_args = TrainingArguments(
    # Output
    output_dir="./sentinel_lora_checkpoints",
    
    # Batch size (critical for OOM prevention)
    per_device_train_batch_size=2,  # Small batch
    gradient_accumulation_steps=4,   # Effective batch = 8
    
    # Training duration
    num_train_epochs=3,  # Adjust based on dataset size
    # max_steps=100,  # Or use fixed steps for testing
    
    # Learning rate
    learning_rate=2e-4,  # Good for LoRA
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # Optimization
    optim="adamw_8bit",  # 8-bit optimizer saves memory
    weight_decay=0.01,
    
    # Precision
    fp16=True,  # T4 is optimized for fp16
    bf16=False,  # T4 doesn't support bf16
    
    # Logging
    logging_steps=10,
    logging_first_step=True,
    
    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,  # Keep only 2 checkpoints
    
    # Misc
    seed=42,
    report_to="none",  # Disable W&B for simplicity
)

In [ ]:
# Create the trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=False,  # True can speed up but may affect quality
)

# Check memory before training
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"💾 GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 🚀 Step 6: Start Training

In [ ]:
# Train!
print("🏋️ Starting training...")
trainer_stats = trainer.train()

# Print training stats
print("\n" + "="*50)
print("📊 Training Complete!")
print("="*50)
print(f"⏱️  Training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"📉 Final loss: {trainer_stats.metrics['train_loss']:.4f}")
if torch.cuda.is_available():
    print(f"💾 Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 💾 Step 7: Save LoRA Adapters to Google Drive

We only save the LoRA adapters (~67MB), not the full model (~16GB).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Save directory
SAVE_PATH = "/content/drive/MyDrive/Sentinel/llama3-vulnerability-lora"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save LoRA adapters only (not the full model!)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ LoRA adapters saved to: {SAVE_PATH}")
print(f"📦 Size: {sum(os.path.getsize(os.path.join(SAVE_PATH, f)) for f in os.listdir(SAVE_PATH)) / 1e6:.1f} MB")

# List saved files
print("\n📁 Saved files:")
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f))
    print(f"   {f}: {size / 1e6:.2f} MB")

## 🔮 Step 8: Test the Fine-tuned Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_code = '''def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    cursor.execute(query)
    return cursor.fetchone()'''

prompt = VULNERABILITY_PROMPT.format(
    instruction="Analyze the following code for security vulnerabilities and provide a fixed version.",
    buggy_code=test_code,
    fixed_code=""  # Model will generate this
)

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("🔍 Model Response:")
print("=" * 50)
print(response)

## 📋 Step 9: Loading the Model Later

To use the fine-tuned model in your Sentinel agent:

```python
from unsloth import FastLanguageModel

# Load base model + adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/path/to/llama3-vulnerability-lora",  # Your saved adapters
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
```

## 🎯 Next Steps

1. **Scale up training**: Use a larger vulnerability dataset (CVEFixes, BigVul)
2. **Integrate with Sentinel**: Load the adapters in your LangGraph agent
3. **Export to GGUF**: For llama.cpp inference

```python
# Export to GGUF for llama.cpp (4-bit quantized)
model.save_pretrained_gguf("llama3-vuln", tokenizer, quantization_method="q4_k_m")
```